In [1]:
import json
from pathlib import Path
from copy import deepcopy
import pandas as pd

# ====== CHANGE PATHS IF NEEDED ======
INPUT_CSV = Path("../Sampling/out/dataset_slices_consensus_7pos_3neg_2hard.csv")
INPUT_JSON = Path("../Sampling/out/dataset_annotations_consensus_7pos_3neg_2hard.json")

OUTPUT_DIR = Path("out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "dataset_target_class.csv"
OUTPUT_JSON = OUTPUT_DIR / "dataset_target_class.json"

print("INPUT_CSV :", INPUT_CSV)
print("INPUT_JSON:", INPUT_JSON)
print("OUTPUT_CSV:", OUTPUT_CSV)
print("OUTPUT_JSON:", OUTPUT_JSON)


INPUT_CSV : ..\Sampling\out\dataset_slices_consensus_7pos_3neg_2hard.csv
INPUT_JSON: ..\Sampling\out\dataset_annotations_consensus_7pos_3neg_2hard.json
OUTPUT_CSV: out\dataset_target_class.csv
OUTPUT_JSON: out\dataset_target_class.json


In [2]:
# Load inputs
df = pd.read_csv(INPUT_CSV)
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print("CSV rows :", len(df))
print("JSON type:", type(data))
print("CSV columns:", len(df.columns))

if isinstance(data, list):
    print("JSON rows:", len(data))
    if len(data):
        print("Sample JSON keys:", list(data[0].keys())[:20])

elif isinstance(data, dict):
    print("Top-level JSON keys count:", len(data.keys()))
    print("Top-level keys sample:", list(data.keys())[:20])
    for k, v in list(data.items())[:10]:
        print(f"Key={k!r}, type={type(v)}")


CSV rows : 3920
JSON type: <class 'dict'>
CSV columns: 10
Top-level JSON keys count: 2
Top-level keys sample: ['images', 'consensus_note']
Key='images', type=<class 'dict'>
Key='consensus_note', type=<class 'str'>


In [3]:
def find_records_container(obj, path=()):
    """
    Find where the per-image records live inside the JSON.

    Returns:
        (container_type, path)
    where:
        container_type is 'list' or 'dict'
        path is a tuple of keys/indexes leading to the records container
    """
    # list of record dicts
    if isinstance(obj, list):
        if len(obj) > 0 and all(isinstance(x, dict) for x in obj):
            # likely record list if many elements contain image_id
            if any("image_id" in x for x in obj):
                return ("list", path)
        for i, x in enumerate(obj):
            found = find_records_container(x, path + (i,))
            if found is not None:
                return found
        return None

    # dict keyed by image_id or nested record-holder
    if isinstance(obj, dict):
        # direct dict-of-records case
        vals = list(obj.values())
        if len(vals) > 0 and all(isinstance(v, dict) for v in vals):
            # either nested dicts already have image_id or outer key is the image id
            if any("image_id" in v for v in vals):
                return ("dict", path)

            # heuristic: keys look like image ids or slice ids
            text_keys = [str(k) for k in obj.keys()]
            if any("LIDC" in k or "slice" in k for k in text_keys[:10]):
                return ("dict", path)

        # search common keys first
        preferred = ["items", "records", "images", "annotations", "data", "samples", "entries", "rows"]
        for key in preferred:
            if key in obj:
                found = find_records_container(obj[key], path + (key,))
                if found is not None:
                    return found

        # fallback recursive search
        for k, v in obj.items():
            found = find_records_container(v, path + (k,))
            if found is not None:
                return found

    return None


def get_by_path(obj, path):
    cur = obj
    for p in path:
        cur = cur[p]
    return cur


def set_by_path(obj, path, new_value):
    if len(path) == 0:
        return new_value
    cur = obj
    for p in path[:-1]:
        cur = cur[p]
    cur[path[-1]] = new_value
    return obj


found = find_records_container(data)
if found is None:
    raise ValueError(
        "Could not locate the per-image records container in INPUT_JSON. "
        "Please inspect the JSON structure."
    )

records_container_type, records_path = found
print("Records container type:", records_container_type)
print("Records path:", records_path)

records_container = get_by_path(data, records_path)
print("Records container length:", len(records_container))


Records container type: dict
Records path: ('images',)
Records container length: 3920


In [4]:
def records_to_lookup(records_container, container_type):
    lookup = {}

    if container_type == "list":
        for item in records_container:
            if not isinstance(item, dict):
                continue
            image_id = item.get("image_id")
            if image_id is not None:
                lookup[image_id] = item

    elif container_type == "dict":
        for k, v in records_container.items():
            if not isinstance(v, dict):
                continue
            rec = deepcopy(v)
            if "image_id" not in rec:
                rec["image_id"] = str(k)
            lookup[rec["image_id"]] = rec

    else:
        raise ValueError("Unsupported records container type")

    return lookup


json_by_image_id = records_to_lookup(records_container, records_container_type)

print("Usable JSON records:", len(json_by_image_id))

missing_in_json = [img_id for img_id in df["image_id"].tolist() if img_id not in json_by_image_id]
print("Rows missing in JSON lookup:", len(missing_in_json))
if missing_in_json[:5]:
    print("Examples:", missing_in_json[:5])


Usable JSON records: 3920
Rows missing in JSON lookup: 0


In [5]:
def get_pos_malignancy_array(json_item):
    """
    Read malignancy information without modifying the selected nodules.
    Search order:
    1) image-level malignancy_array
    2) first available nested malignancy_array under known keys
    """
    if json_item is None:
        return None

    arr = json_item.get("malignancy_array")
    if isinstance(arr, list):
        return arr

    for key in ["consensus_nodules", "consensus_nodules_cropped"]:
        nods = json_item.get(key)
        if isinstance(nods, list):
            for nod in nods:
                if isinstance(nod, dict):
                    arr = nod.get("malignancy_array")
                    if isinstance(arr, list):
                        return arr

    return None


def assign_target(sample_type, json_item):
    if sample_type in {"neg", "hard_neg"}:
        return {
            "keep": True,
            "target_class": "benign",
            "target_class_id": 0,
            "malignancy_avg": None,
            "skip_reason": None,
        }

    if sample_type == "pos":
        arr = get_pos_malignancy_array(json_item)

        if not isinstance(arr, list):
            return {
                "keep": False,
                "target_class": None,
                "target_class_id": None,
                "malignancy_avg": None,
                "skip_reason": "missing_malignancy_array",
            }

        if len(arr) < 3:
            return {
                "keep": False,
                "target_class": None,
                "target_class_id": None,
                "malignancy_avg": None,
                "skip_reason": "malignancy_array_len_lt_3",
            }

        avg = float(sum(arr) / len(arr))

        if avg < 2.5:
            return {
                "keep": True,
                "target_class": "benign",
                "target_class_id": 0,
                "malignancy_avg": avg,
                "skip_reason": None,
            }
        elif avg >= 4.0:
            return {
                "keep": True,
                "target_class": "malignant",
                "target_class_id": 1,
                "malignancy_avg": avg,
                "skip_reason": None,
            }
        else:
            return {
                "keep": False,
                "target_class": None,
                "target_class_id": None,
                "malignancy_avg": avg,
                "skip_reason": "ambiguous_avg_range",
            }

    return {
        "keep": False,
        "target_class": None,
        "target_class_id": None,
        "malignancy_avg": None,
        "skip_reason": f"unsupported_sample_type:{sample_type}",
    }


In [6]:
# Build outputs
kept_csv_rows = []
kept_json_lookup = {}
skipped_rows = []

for _, row in df.iterrows():
    row_dict = row.to_dict()
    image_id = row_dict["image_id"]
    sample_type = str(row_dict.get("sample_type", "")).strip()
    json_item = json_by_image_id.get(image_id)

    result = assign_target(sample_type, json_item)

    if result["keep"]:
        # CSV: keep every original column, only append target fields
        out_row = deepcopy(row_dict)
        out_row["target_class"] = result["target_class"]
        out_row["target_class_id"] = result["target_class_id"]
        out_row["malignancy_avg"] = result["malignancy_avg"]
        kept_csv_rows.append(out_row)

        # JSON record: keep all original information, append target fields
        if json_item is not None:
            out_item = deepcopy(json_item)
        else:
            out_item = {"image_id": image_id}

        out_item["target_class"] = result["target_class"]
        out_item["target_class_id"] = result["target_class_id"]
        out_item["malignancy_avg"] = result["malignancy_avg"]

        kept_json_lookup[image_id] = out_item
    else:
        skipped_rows.append({
            "image_id": image_id,
            "sample_type": sample_type,
            "skip_reason": result["skip_reason"],
            "malignancy_avg": result["malignancy_avg"],
        })

out_df = pd.DataFrame(kept_csv_rows)

print("Kept rows   :", len(out_df))
print("Skipped rows:", len(skipped_rows))
print()
if len(out_df):
    print("Sample type counts:")
    print(out_df["sample_type"].value_counts(dropna=False))
    print()
    print("Target class counts:")
    print(out_df["target_class"].value_counts(dropna=False))


Kept rows   : 2385
Skipped rows: 1535

Sample type counts:
sample_type
neg         912
hard_neg    881
pos         592
Name: count, dtype: int64

Target class counts:
target_class
benign       2057
malignant     328
Name: count, dtype: int64


In [7]:
# Rebuild full JSON structure while preserving all top-level information
full_json_out = deepcopy(data)

if records_container_type == "list":
    new_records_container = list(kept_json_lookup.values())
else:
    # preserve dict-by-image_id style for the records container
    new_records_container = {img_id: rec for img_id, rec in kept_json_lookup.items()}

full_json_out = set_by_path(full_json_out, records_path, new_records_container)

# Add a small summary block without removing anything
summary_block = {
    "source_csv": str(INPUT_CSV),
    "source_json": str(INPUT_JSON),
    "output_csv": str(OUTPUT_CSV),
    "output_json": str(OUTPUT_JSON),
    "kept_count": int(len(out_df)),
    "skipped_count": int(len(skipped_rows)),
    "rules": {
        "neg_and_hard_neg": "benign",
        "pos_len_lt_3": "skip",
        "pos_avg_lt_2.5": "benign",
        "pos_avg_ge_4.0": "malignant",
        "otherwise": "skip"
    }
}

if isinstance(full_json_out, dict):
    full_json_out["target_class_build_summary"] = summary_block


In [8]:
# Save outputs
out_df.to_csv(OUTPUT_CSV, index=False)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(full_json_out, f, indent=2)

print("Saved:")
print(" -", OUTPUT_CSV)
print(" -", OUTPUT_JSON)


Saved:
 - out\dataset_target_class.csv
 - out\dataset_target_class.json


In [9]:
# Quick preview
print("CSV preview:")
display(out_df.head(10))

print("\nJSON preview:")
if isinstance(full_json_out, dict):
    print("Top-level keys sample:", list(full_json_out.keys())[:20])
    recs = get_by_path(full_json_out, records_path)
    if isinstance(recs, dict) and len(recs):
        sample_key = next(iter(recs.keys()))
        preview = recs[sample_key]
        print("Sample record key:", sample_key)
        print({
            "image_id": preview.get("image_id"),
            "sample_type": preview.get("sample_type"),
            "target_class": preview.get("target_class"),
            "target_class_id": preview.get("target_class_id"),
            "malignancy_avg": preview.get("malignancy_avg"),
        })
    elif isinstance(recs, list) and len(recs):
        preview = recs[0]
        print({
            "image_id": preview.get("image_id"),
            "sample_type": preview.get("sample_type"),
            "target_class": preview.get("target_class"),
            "target_class_id": preview.get("target_class_id"),
            "malignancy_avg": preview.get("malignancy_avg"),
        })
else:
    print(type(full_json_out))


CSV preview:


,image_id,patient_id,slice_index,z_position,dicom_path,sop_instance_uid,sample_type,has_nodule,num_consensus_nodules,num_non_nodule_marks,target_class,target_class_id,malignancy_avg
0,LIDC-IDRI-0001__slice_0020,LIDC-IDRI-0001,20,-290.0,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.125539044472...,neg,0,0,0,benign,0,NaN
1,LIDC-IDRI-0001__slice_0039,LIDC-IDRI-0001,39,-242.5,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.588464900616...,hard_neg,0,0,4,benign,0,NaN
2,LIDC-IDRI-0001__slice_0087,LIDC-IDRI-0001,87,-122.5,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.499837844441...,pos,1,1,0,malignant,1,4.75
3,LIDC-IDRI-0001__slice_0089,LIDC-IDRI-0001,89,-117.5,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.824843590991...,pos,1,1,0,malignant,1,4.75
4,LIDC-IDRI-0001__slice_0091,LIDC-IDRI-0001,91,-112.5,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.261151233960...,pos,1,1,0,malignant,1,4.75
5,LIDC-IDRI-0002__slice_0020,LIDC-IDRI-0002,20,-309.5,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.138792614529...,neg,0,0,0,benign,0,NaN
6,LIDC-IDRI-0002__slice_0032,LIDC-IDRI-0002,32,-294.5,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.226735728123...,hard_neg,0,0,2,benign,0,NaN
7,LIDC-IDRI-0003__slice_0020,LIDC-IDRI-0003,20,-329.0,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.137278294478...,neg,0,0,0,benign,0,NaN
8,LIDC-IDRI-0003__slice_0038,LIDC-IDRI-0003,38,-284.0,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.337740860594...,hard_neg,0,0,1,benign,0,NaN
9,LIDC-IDRI-0003__slice_0077,LIDC-IDRI-0003,77,-186.5,E:\DATASET\manifest-1775741638894\LIDC-IDRI\LI...,1.3.6.1.4.1.14519.5.2.1.6279.6001.296257982882...,pos,1,1,0,malignant,1,4.25



JSON preview:
Top-level keys sample: ['images', 'consensus_note', 'target_class_build_summary']
Sample record key: LIDC-IDRI-0001__slice_0020
{'image_id': 'LIDC-IDRI-0001__slice_0020', 'sample_type': 'neg', 'target_class': 'benign', 'target_class_id': 0, 'malignancy_avg': None}


In [10]:
# Optional: inspect skipped rows
skip_df = pd.DataFrame(skipped_rows)
if len(skip_df):
    print(skip_df["skip_reason"].value_counts(dropna=False))
    display(skip_df.head(20))
else:
    print("No skipped rows.")


skip_reason
ambiguous_avg_range          789
malignancy_array_len_lt_3    746
Name: count, dtype: int64


,image_id,sample_type,skip_reason,malignancy_avg
0,LIDC-IDRI-0002__slice_0182,pos,malignancy_array_len_lt_3,NaN
1,LIDC-IDRI-0002__slice_0184,pos,malignancy_array_len_lt_3,NaN
2,LIDC-IDRI-0002__slice_0186,pos,malignancy_array_len_lt_3,NaN
3,LIDC-IDRI-0003__slice_0066,pos,malignancy_array_len_lt_3,NaN
4,LIDC-IDRI-0003__slice_0084,pos,ambiguous_avg_range,3.250000
5,LIDC-IDRI-0005__slice_0077,pos,ambiguous_avg_range,2.750000
6,LIDC-IDRI-0005__slice_0080,pos,ambiguous_avg_range,2.750000
7,LIDC-IDRI-0005__slice_0088,pos,malignancy_array_len_lt_3,NaN
8,LIDC-IDRI-0006__slice_0054,pos,malignancy_array_len_lt_3,NaN
9,LIDC-IDRI-0006__slice_0070,pos,malignancy_array_len_lt_3,NaN
